## Before you start:
**Tools → Settings → Editor → completions / suggestions / linting → disable**

## Task 1

Write 2 implementations of the `increment()` function that **increments the global variable `counter` by 1**:

**with and without** (!) Python syntactic sugar. You may not change the function signature.

**1.1: with Python syntactic sugar**

In [1]:
counter = 0

def increment():
    global counter
    counter += 1

increment()
increment()
assert counter == 2, 'try again'
print(f'{counter=} -- great!')

counter=2 -- great!


**1.2: without Python syntactic sugar**

In [2]:
counter = 0

def increment():
    global counter
    counter = counter + 1

increment()
increment()
assert counter == 2, 'try again'
print(f'{counter=} -- great!')

counter=2 -- great!


Alternative:

In [3]:
counter = 0

def increment():
    globals()['counter'] = globals()['counter'].__add__(1)

increment()
increment()
assert counter == 2, 'try again'
print(f'{counter=} -- great!')

counter=2 -- great!


## Task 2

Import **only the `sqrt` function** from the `math` module and execute sqrt(169).  
You may not execute `import math`.

Provide 2 solutions.

Solution 1:

In [4]:
from math import sqrt
result = sqrt(169)

assert result == 13
print(f"{result=}")

result=13.0


Solution 2:

In [5]:
sqrt = __import__("math").sqrt
result = sqrt(169)

assert result == 13
print(f"{result=}")

result=13.0


## Task 3

Dynamic import and reload.

1. Create a module `mod.py`:

In [6]:
%%writefile mod.py
msg = "A"

Writing mod.py


2. Import it and print `msg`:

In [7]:
import mod

print(mod.msg)

A


3. Change `msg` to `B` in the file:

In [8]:
%%writefile mod.py
msg = "B"

Overwriting mod.py


4. Without restarting the notebook session, print the new value of `msg`:

In [9]:
import importlib

importlib.reload(mod)
print(mod.msg)

B


## Task 4

You have a directory `pkg`:

In [1]:
!mkdir -p pkg

In [2]:
%%writefile pkg/m1.py
pi = 3.1415_92_65
_e = 2.7
__i = -1

Writing pkg/m1.py


```
pkg/
└── m1.py
```

Below are cells for your code; the task follows

### **First way**: via a module from the CPython standard library

In [3]:
%%writefile pkg/__init__.py
import importlib

m1 = importlib.import_module("pkg.m1")

# reference to original value, not re-created
pi = m1.pi

__all__ = ['pi']

Writing pkg/__init__.py


In [4]:
from pkg import *
pi

3.14159265

### **Second way**: in one line without extra modules

In [5]:
%%writefile pkg/__init__.py
from .m1 import *

Overwriting pkg/__init__.py


In [6]:
from pkg import *
pi

3.14159265

### Task text:
You may not re-create the values `pi`, `_e`, `__i` or use them directly in the import.  
You must change the structure of the `pkg` package / contents of its modules so that the following code runs correctly:

In [ ]:
from pkg import *
pi

3.14159265

**Important!**  
Whenever you update any data in the project directory, you must reload the ipynb session:  
`Runtime --> Restart Session` and re-run the necessary task cells,  
otherwise the results may be incorrect for you.

## Task 5

After correctly solving **Task 4**, you need to:
- modify `pkg`
- complete the code below

so that you can reach `__i`.  
You may not re-create the values `pi`, `_e`, `__i` or use them directly in the import.  
If you restart the session, you must re-run the cells for **Task 4** before solving **Task 5**.

In [7]:
%%writefile pkg/__init__.py
import importlib

m1 = importlib.import_module("pkg.m1")

# reference to original value, not re-created
pi = m1.pi
_e = m1._e
__i = m1.__i

__all__ = ['pi', '_e', '__i']

Overwriting pkg/__init__.py


In [8]:
import importlib, pkg
importlib.reload(pkg)

<module 'pkg' from 'd:\\Projects\\mipt-python-course\\pkg\\__init__.py'>

In [9]:
from pkg import *
__i

-1

## Task 6

Mutable closure. Why does this code behave unexpectedly? Fix it.

The code behaves unexpectedly because:
1) the accs array contains a set of values, not references to accumulator functions
2) the inner function captures the same mutable list and the same variable i, which changes in the loop. At the time of the call, all accumulators have i = 2

In [12]:
def create_accumulators():
    accs = []
    values = []
    for i in range(3):
        def accumulator(x, idx=i):
            values[idx] += x
            return values[idx]
        values.append(0)
        accs.append(accumulator)
    return accs

acc_list = create_accumulators()
print(acc_list[0](10))  # Expected 10, but an error occurs

10


## Task 7
When the notebook session is restarted, the solution starts from scratch

### 7.1: Restore the `print` functionality without using `del`

In [1]:
print = 1
print

1

In [2]:
import builtins

print = builtins.print
print

<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [3]:
print("Hello, world!")

Hello, world!


### 7.2: Delete the object `print`, then restore its functionality

In [12]:
import builtins

saved_print = builtins.print
del print
del builtins.print

In [13]:
print("Hello, world!")

NameError: name 'print' is not defined

In [15]:
builtins.print = saved_print
print = builtins.print

print("Hello, world!")

Hello, world!


## Task 8

Closure with mutable state. Create a counter function that remembers the number of calls across different instances:

In [7]:
def make_shared_counter(count=[0]):
    def counter():
        count[0] += 1
        return count[0]
    return counter

c1 = make_shared_counter()
c2 = make_shared_counter()

print(c1())
print(c2())
print(c1())

1
2
3


## Task 9

Complete the code so that the `outer` function returns **a dictionary with three closures**: `add()`, `mul()`, `get()` — all working with the same closed variable `value`.

In [12]:
def outer(val=0):
    def add(x):
        nonlocal val
        val += x
    
    def mul(x):
        nonlocal val
        val *= x
    
    def get():
        nonlocal val
        return val
    
    return {'add': add, 'mul': mul, 'get': get}


obj = outer(10)
obj['add'](5)
obj['mul'](2)
assert obj['get']() == 30
print(f"{obj['get']()=}")

obj['get']()=30


## Task 10

Create a closure that takes a function and returns a new function with result caching (memoization).

*Theoretical note:*

The `memoize` function accepts another function `func` and creates an inner closure with a dictionary `cache` for storing results.

In the example with the `fib` function (Fibonacci numbers), memoization reduces the number of recursive calls from exponential to linear, since each `n` is computed only once.

Thus, memoization saves time at the expense of memory — a classic "time vs. memory" optimization — and is especially useful for functions with expensive computations and repeated inputs.

Algorithm for solution:

- The memoize function accepts another function func and creates an inner closure with a dictionary cache for storing results.

- The inner wrapper function checks whether a result for the given input argument (x) is already in the cache dictionary.

- If it is, the cached result is returned and no recomputation occurs.

- If not, the original function func(x) is called, the result is saved in cache and returned.


In [16]:
def memoize(func):
    cache = {}
    def wrapper(x):
        if (x in cache):
            return cache[x]
        result = func(x)
        cache[x] = result
        return result
    return wrapper

@memoize
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

print(fib(10))  # 55

55
